# Planted Noisy kXOR - Kikuchi Guiding State

Prepare the guiding state for a kXOR instance $\mathcal{I}$ with
Kikuchi parameter $\ell$.

References:
    [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1)
    Section 4.4.1, Theorem 4.15.

Native counterpart of the complete frozen Qualtran v0.7.0 notebook at
`bcb17bb25360ed68ce3bc084645db3e929799389`, blob
`ca6e8c12a2c0f2e387609045996675e2738b5205`. Copyright 2024 Google LLC,
adapted under Apache-2.0. The source scope-state name maps to
`ScopeGuidingState` and its sparse dependency to `SparseRotationPreparation`.

The notebook reproduces declarations, symbolic call graphs and QEC estimates.
Full root physical execution remains open at the source sorting and protected
Core bounded-register boundary. `KIKUCHI_GUIDING_STATE_FILE_AUDIT.md` records
this and the omitted phase-gradient port. Executable probabilistic uncomputation
uses `circuit.append(op)` and has separate primitive evidence.


In [ ]:
from collections.abc import Callable
from typing import cast

from IPython import display as notebook_display

from flagquantum.algorithms.kikuchi_guiding_state import (
    example_guiding_state,
    example_scope_guiding_state,
    symbolic_count_guiding_state,
    symbolic_guiding_state,
    symbolic_scope_guiding_state,
)
from flagquantum.core.ir.program.signatures import Signature
from flagquantum.drawer.call_graph import export_call_graph_dot, render_call_graph_svg
from flagquantum.drawer.signature_graph import export_signature_graph_dot

SVG = cast(Callable[[bytes], object], notebook_display.SVG)
display = cast(Callable[[object], None], notebook_display.display)


def show_signature(name: str, signature: Signature) -> None:
    if all(
        type(register.bitsize) is int
        and all(type(size) is int for size in register.shape_symbolic)
        for register in signature
    ):
        dot = export_signature_graph_dot(signature, name)
    else:
        rows = {}
        for register in signature:
            rows[register.name + " width"] = str(register.bitsize)
            rows[register.name + " shape"] = str(register.shape_symbolic)
            rows[register.name + " side"] = str(register.side.name)
            rows[register.name + " encoding"] = str(register.dtype.encoding)
            rows.update({key: str(value) for key, value in register.dtype.parameters})
        dot = export_call_graph_dot(name, {}, annotations={name: rows})
    display(SVG(render_call_graph_svg(dot)))


## `SimpleGuidingState`
Prepare the guiding state for $\ell = k$.

Given an kXOR instance $\mathcal{I}$, prepare the guiding state for
parameter $\ell = k$ (i.e. $c = 1$), defined in Eq 134:
    $$
    |\phi\rangle
    \propto
    |\Gamma^k(\mathcal{A})\rangle
    =
    \frac{1}{\sqrt{\tilde{m}}}
    \sum_{S \in {[n] \choose k}} B_\mathcal{I}(S) |S\rangle
    $$

Here, $\tilde{m}$ is the number of constraints in the input instance $\mathcal{I}$,
and $\mathcal{A} = \sqrt{\frac{{n\choose k}}{\tilde{m}}} \mathcal{I}$.

This bloq has a gate cost of $O(\tilde{m} \log n)$ (see Eq 142 and paragraph below).

#### Parameters
 - `inst`: the kXOR instance $\mathcal{I}$.
 - `eps`: Precision of the prepared state (defaults to 1e-6). 

#### Registers
 - `S`: a scope of $k$ variables, each in $[n]$. 

#### References
 - [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1). Equation 134.


In [ ]:
# Scope preparation is imported through its factory in setup.


### Example Instances

In [ ]:
simple_guiding_state_symb = symbolic_scope_guiding_state()


In [ ]:
simple_guiding_state = example_scope_guiding_state()


#### Graphical Signature

In [ ]:
show_signature("simple_guiding_state_symb", simple_guiding_state_symb.signature)
show_signature("simple_guiding_state", simple_guiding_state.signature)


### Call Graph

In [ ]:
simple_guiding_state_symb_g = simple_guiding_state_symb.source_graph(
    root="ScopeGuidingState", preparation="SparseRotationPreparation"
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                simple_guiding_state_symb_g.root,
                simple_guiding_state_symb_g.direct_calls,
            )
        )
    )
)
simple_guiding_state_symb_sigma = simple_guiding_state_symb_g.leaf_counts
display(simple_guiding_state_symb_sigma)


## `GuidingState`
Prepare a guiding state for a kXOR instance with parameter $\ell$.

Given an kXOR instance $\mathcal{I}$, and parameter $\ell$ (a multiple of $k$),
we want to prepare the unit-length guiding state $|\mathbb{\Psi}\rangle$ (Eq 135):

    $$
    |\mathbb{\Psi}\rangle
    \propto
    |\Gamma^\ell(\mathcal{A})\rangle
    \propto
    \sum_{T \in {[n] \choose \ell}}
    \sum_{\{S_1, \ldots, S_c\} \in \text{Part}_k(T)}
    \left(
    \prod_{j = 1}^c B_{\mathcal{I}}(S)
    \right)
    |T\rangle
    $$

This bloq prepares the state (Eq 136):
    $$ \beta |\mathbb{\Psi}\rangle |0^{\ell \log \ell + 3}\rangle
       + |\perp\rangle |1\rangle
    $$
where $\beta \ge \Omega(1 / \ell^{\ell/2})$,
and $\tilde{m}$ is the number of constraints in $\mathcal{I}$.

This has a gate cost of $O(\ell \tilde{m} \log n)$.

#### Parameters
 - `inst`: the kXOR instance $\mathcal{I}$.
 - `ell`: the Kikuchi parameter $\ell$.
 - `amplitude_good_part`: (optional) the amplitude $\beta$ of the guiding state $|\Psi\rangle$ Defaults to $\beta = 0.99 / \ell^{\ell/2}$.
 - `eps`: Precision of the prepared state (defaults to 1e-6). 

#### Registers
 - `T`: $\ell$ indices each in $[n]$.
 - `ancilla`: (entangled) $\ell\log\ell+3$ ancilla qubits used for state preparation. The all zeros state of the ancilla is the good subspace. 

#### References
 - [Quartic quantum speedups for planted inference](https://arxiv.org/abs/2406.19378v1). Section 4.4.1 "Preparing the guiding state", Theorem 4.15. Eq 136.


In [ ]:
# The full guiding-state factory belongs to Algorithms.


### Example Instances

In [ ]:
guiding_state_symb_c = symbolic_count_guiding_state()


In [ ]:
guiding_state_symb = symbolic_guiding_state()


In [ ]:
guiding_state = example_guiding_state()


#### Graphical Signature

In [ ]:
show_signature("guiding_state_symb_c", guiding_state_symb_c.signature)
show_signature("guiding_state_symb", guiding_state_symb.signature)
show_signature("guiding_state", guiding_state.signature)


### Call Graph

In [ ]:
guiding_state_symb_c_g = guiding_state_symb_c.source_graph(
    root="GuidingState",
    preparation="ScopeGuidingState",
    sorting="SortInPlace",
    duplicates="HasDuplicates",
    uncompute="ProbabilisticUncompute",
    flag_predicate="MultiControlX",
    flag_x="XGate",
)
display(
    SVG(
        render_call_graph_svg(
            export_call_graph_dot(
                guiding_state_symb_c_g.root, guiding_state_symb_c_g.direct_calls
            )
        )
    )
)
guiding_state_symb_c_sigma = guiding_state_symb_c_g.leaf_counts
display(guiding_state_symb_c_sigma)


In [ ]:
model_categories = guiding_state.resource_gates()
symbolic_model_categories = guiding_state_symb_c.resource_gates()
assert model_categories.toffoli == 132
assert model_categories.cswap == 192
assert model_categories.and_bloq == 352
assert model_categories.clifford == 4173
display(model_categories)
display(symbolic_model_categories)
